In [53]:
import json
import os
import numpy as np
import pickle
# Set project root so relative paths (e.g. routing_performance.json) work
ROOT = "/data/gpfs/projects/punim2662/routing_system"
os.chdir(ROOT)
print(f"Working directory: {os.getcwd()}")

Working directory: /data/gpfs/projects/punim2662/routing_system


In [54]:
with open('routing_performance.json', 'r') as f:
    data = json.load(f)
results_list = data['results']
# Find unique backends
unique_backends = set()
for result in results_list:
    if result['backend'] is not None:
        unique_backends.add(result['backend'])
ttfts_dict = {}
tbt_dict = {}
time_to_choose_backend_dict = {}
for backend in unique_backends:
    ttfts_dict[backend] = []
    time_to_choose_backend_dict[backend] = []
    tbt_dict[backend] = []
for result in results_list:
    if result['backend'] is not None:
        backend = result['backend']
        ttfts_dict[backend].append(result['ttft_ms'])
        time_to_choose_backend_dict[backend].append(result['time_to_choose_backend_ms'])
        tbt_dict[backend] += result['tbts_ms']
for backend in unique_backends:
    print(f'{backend} length: {len(ttfts_dict[backend])}')
    print(f'avg ttft for {backend}: {np.mean(ttfts_dict[backend])}')
    print(f'avg time to choose backend for {backend}: {np.mean(time_to_choose_backend_dict[backend])}')
    print(f'avg tbt for {backend}: {np.mean(tbt_dict[backend])} \n')

print(f'total requests: {len(results_list)}')
print(f'average ttft: {np.mean([np.mean(ttfts_dict[backend]) for backend in unique_backends])}')
print(f'average tbt: {np.mean([np.mean(tbt_dict[backend]) for backend in unique_backends])}')
print(f'average time to choose backend: {np.mean([np.mean(time_to_choose_backend_dict[backend]) for backend in unique_backends])}')


http://127.0.0.1:8001 length: 308
avg ttft for http://127.0.0.1:8001: 41225.14454575328
avg time to choose backend for http://127.0.0.1:8001: 632.4577922449
avg tbt for http://127.0.0.1:8001: 107.39935441831848 

http://127.0.0.1:8002 length: 814
avg ttft for http://127.0.0.1:8002: 1747.9439669217759
avg time to choose backend for http://127.0.0.1:8002: 308.8882063453701
avg tbt for http://127.0.0.1:8002: 41.92572428903596 

total requests: 1716
average ttft: 21486.54425633753
average tbt: 74.66253935367722
average time to choose backend: 470.67299929513507


In [55]:

import pandas as pd
def contains_chinese(text: str) -> bool:
    for char in text:
        code = ord(char)
        # Common CJK ranges
        if (
            0x4E00 <= code <= 0x9FFF or   # CJK Unified Ideographs
            0x3400 <= code <= 0x4DBF or   # CJK Unified Ideographs Extension A
            0x20000 <= code <= 0x2A6DF or # Extension B
            0x2A700 <= code <= 0x2B73F or # Extension C
            0x2B740 <= code <= 0x2B81F or # Extension D
            0x2B820 <= code <= 0x2CEAF or # Extension E
            0xF900 <= code <= 0xFAFF      # CJK Compatibility Ideographs
        ):
            return True
    return False

with open('router_model/datasets/routerbench_0shot.pkl', 'rb') as f:
    data = pickle.load(f)

# remove chinese from data
english_data = data[data.apply(lambda row: not contains_chinese(row['prompt']), axis=1)]
mistral_data = data['mistralai/mistral-7b-chat']
llama_70b_data = data['zero-one-ai/Yi-34B-Chat']
mixtral_data = data['WizardLM/WizardLM-13B-V1.2']
clean_data = []

for index, row in english_data.iterrows():
    new_row = {
        'prompt': row['prompt'],
        'mistralai/mistral-7b-chat': row['mistralai/mistral-7b-chat'],
        'zero-one-ai/Yi-34B-Chat': row['zero-one-ai/Yi-34B-Chat'],
        'WizardLM/WizardLM-13B-V1.2': row['WizardLM/WizardLM-13B-V1.2']
    }
    clean_data.append(new_row)

# convert data to pandas dataframe
clean_data = pd.DataFrame(clean_data)
print(clean_data.keys())
# save clean data
clean_data.to_pickle('router_model/datasets/routerbench_0shot_clean.pkl')

/tmp/ipykernel_9590/2524540695.py:19: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  data = pickle.load(f)


Index(['prompt', 'mistralai/mistral-7b-chat', 'zero-one-ai/Yi-34B-Chat',
       'WizardLM/WizardLM-13B-V1.2'],
      dtype='str')
